# Turtle Breakout 06 — protective put at the entry strike, and calls instead of the perpetual

Pre-registration: `discussions/Turtle_Breakout/008_claude_tb06_preregistration.md`. Code: `src/data_engineering/modeling/turtle_options.py`.

The daily turtle (TB 01 rules) on BTC and ETH. Option tenor: the first Friday 08:00 UTC at least **38 days** after entry (mean hold of the $R\ge3$ trades in the TB 04 discovery set).

| | strategy |
|---|---|
| S0 | perpetual (TB 01) |
| S1 | perpetual + put struck at the entry price, sold when the turtle exits (or settled at expiry) |
| S2 | call struck at the entry price instead of the perpetual, sold when the turtle exits, rolled at expiry |
| S3 | call struck one ATR above the entry |

Options priced with Black-76 at Deribit's DVOL (30-day ATM implied vol), bought at $\sigma(1+h)$ and sold at $\sigma(1-h)$, $h=2\%$; Deribit option fees. Results per trade in $R$ = 2 ATR at entry.

Environment: `TB01_DATA_DIR`, `TB06_SYNTHETIC=1`.

In [ ]:
import os, sys, json, time, subprocess, tempfile, warnings
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_options as to, turtle_trade_features as tf

SYNTHETIC = os.getenv('TB06_SYNTHETIC', '0') == '1'
P = tb.Params(); O = to.OParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb06-')) / '06'; O = replace(O, n_boot=300)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '06'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    DVOL_DIR = Path(os.environ['DERIBIT_DATA_ROOT']) / 'history' / 'dvol'
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_options.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print(json.dumps(to.registration(O), indent=1))

## 1. Data, turtle trades and option legs

In [ ]:
T0 = time.time(); D, DV, TR = {}, {}, {}
for i, a in enumerate(to.ASSETS):
    if SYNTHETIC:
        b = tb.synthetic_5m(2100, seed=30 + i, trend=0.01, start='2020-09-01'); f = tb.synthetic_funding(b, 0.0001)
        dv = to.synthetic_dvol(pd.Timestamp('2021-03-24', tz='UTC'), b.index[-1], 0.6 + 0.1 * i)
    else:
        b = tb.bars_5m(pd.read_parquet(DATA / f'{a}USDT_um_1m.parquet', columns=['open_time', 'open', 'high', 'low', 'close']))
        f = tb.funding_events(pd.read_parquet(DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'))
        dv = to.load_dvol(DVOL_DIR / f'{a}_dvol_1h.csv')
    D[a] = tb.prepare_asset(a, b, f, P); DV[a] = dv
    START = max(tb.COMMON_START, dv.index[0] + pd.Timedelta(days=2))
    TR[a] = tf.run_trades(D[a], P, START)
    print(a, 'DVOL', dv.index[0].date(), '->', dv.index[-1].date(), '| trades from', START.date(), ':', len(TR[a]))
END = min(d.end[-1] for d in D.values())
X = {h: pd.concat([to.run_asset(D[a], DV[a], TR[a], O, h) for a in to.ASSETS], ignore_index=True) for h in (O.half_spread, 0.0, 0.05)}
XP = X[O.half_spread]
print(f'{len(XP)} trades, {time.time() - T0:.0f}s'); display(XP.head(3))

## 2. Per-trade results (R)

In [ ]:
COLS = {'S0 perp': 'perp_R', 'S1 perp + put (entry strike)': 'perp_put_R', 'S1b perp + put held to expiry': 'perp_put_hold_R',
        'S2 call ATM (rolled)': 'callATM_R', 'S2b call ATM (no roll)': 'callATM_noroll_R', 'S3 call +1 ATR (rolled)': 'callOTM_R',
        'S3b call +1 ATR (no roll)': 'callOTM_noroll_R'}
def per_trade(Xh):
    return pd.DataFrame({k: {'trades': len(Xh), 'win_rate': float((Xh[c] > 0).mean()), 'mean_R': float(Xh[c].mean()), 'median_R': float(Xh[c].median()),
                             'sum_R': float(Xh[c].sum()), 'worst_R': float(Xh[c].min()), 'best_R': float(Xh[c].max()),
                             'R_from_top5': float(Xh[c].nlargest(5).sum())} for k, c in COLS.items()}).T
PT = per_trade(XP); display(PT.round(3))
print('Put leg alone: mean premium', round(XP.put_premium_R.mean(), 3), 'R; mean put P&L', round(XP.put_R.mean(), 3), 'R; pays (>0) on',
      f"{(XP.put_R > 0).mean():.0%} of trades; expired before the turtle exit on {(XP.put_how == 'expired').mean():.0%}")
print('Call ATM: mean first premium', round(XP.callATM_premium_R.mean(), 3), 'R; trades needing a roll', f"{(XP.callATM_legs > 1).mean():.0%}")
SENS = pd.concat({f'h={h:.0%}': per_trade(Xh)[['mean_R', 'sum_R']] for h, Xh in X.items()}, axis=1); print('Spread sensitivity'); display(SENS.round(3))
bins = pd.cut(XP.perp_R, [-np.inf, -0.5, 0, 1, 3, np.inf])
display(XP.groupby(bins, observed=True)[['perp_R', 'put_R', 'perp_put_R', 'callATM_R', 'callOTM_R']].agg(['count', 'mean']).round(3))

## 3. Pre-registered comparisons on daily cumulative-R curves (BTC + ETH)

In [ ]:
START = max(pd.Timestamp(XP.entry_time.min()), tb.COMMON_START)
days = to.daily_days(START, END)
CURVE = {k: sum(to.daily_curve(D[a], DV[a], XP[XP.asset == a], k, O, O.half_spread, days) for a in to.ASSETS)
         for k in ('perp', 'perp_put', 'callATM', 'callOTM')}
q = O.alpha / 2
Q = {'Q1 S1 perp+put vs S0 perp': to.sharpe_diff_R(CURVE['perp_put'], CURVE['perp'], O, q, O.seed + 1),
     'Q2 S2 call ATM vs S0 perp': to.sharpe_diff_R(CURVE['callATM'], CURVE['perp'], O, q, O.seed + 2),
     'S3 call +1 ATR vs S0 (reported)': to.sharpe_diff_R(CURVE['callOTM'], CURVE['perp'], O, q, O.seed + 3)}
display(pd.DataFrame(Q).T.round(3))
MD = {'Q1 per-trade mean R, S1 - S0': to.mean_diff_month(XP.perp_put_R.to_numpy(), XP.perp_R.to_numpy(), pd.DatetimeIndex(XP.entry_time), O, q, O.seed + 4),
      'Q2 per-trade mean R, S2 - S0': to.mean_diff_month(XP.callATM_R.to_numpy(), XP.perp_R.to_numpy(), pd.DatetimeIndex(XP.entry_time), O, q, O.seed + 5)}
display(pd.DataFrame(MD).T.round(3))
CUR = pd.DataFrame({k: {'total_R': float(c.iloc[-1]), 'max_drawdown_R': to.max_dd(c), 'sharpe_daily': float(c.diff().mean() / c.diff().std() * np.sqrt(365))} for k, c in CURVE.items()}).T
display(CUR.round(3))
DECISION = {'Q1_put_hedge': Q['Q1 S1 perp+put vs S0 perp']['result'], 'Q2_call_only': Q['Q2 S2 call ATM vs S0 perp']['result'],
            'trades': len(XP), 'window': [str(START), str(END)]}
print(json.dumps(DECISION, indent=1))

## 4. Years

In [ ]:
yr = pd.to_datetime(XP.exit_time).dt.year
YR = XP.groupby(yr)[['perp_R', 'perp_put_R', 'callATM_R', 'callOTM_R']].sum().assign(trades=XP.groupby(yr).size())
display(YR.round(2))

## 5. Figures

In [ ]:
cols = {'perp': ('#1f5aa6', 'S0 perpetual'), 'perp_put': ('#c2571a', 'S1 perpetual + put'), 'callATM': ('#3a7d44', 'S2 call ATM'), 'callOTM': ('#8a4fa3', 'S3 call +1 ATR')}
fig, axes = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True, gridspec_kw={'height_ratios': [2.2, 1]})
for k, (c, lab) in cols.items():
    axes[0].plot(CURVE[k].index, CURVE[k], color=c, lw=1.4, label=lab); axes[1].plot(CURVE[k].index, CURVE[k] - CURVE[k].cummax(), color=c, lw=1)
axes[0].set_ylabel('cumulative R (BTC + ETH)'); axes[0].legend(frameon=False, fontsize=8); axes[1].set_ylabel('drawdown (R)')
axes[0].set_title('Daily turtle: perpetual vs protective put vs calls (one unit per trade, no compounding)', loc='left', fontsize=11)
for ax in axes: ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, k, c in ((axes[0], 'perp_put_R', '#c2571a'), (axes[1], 'callATM_R', '#3a7d44')):
    ax.scatter(XP.perp_R, XP[k], s=14, color=c, alpha=0.8); lim = [XP.perp_R.min() - 0.5, XP.perp_R.max() + 0.5]
    ax.plot(lim, lim, color='k', lw=0.7); ax.axhline(0, color='0.6', lw=0.6); ax.axvline(0, color='0.6', lw=0.6)
    ax.set_xlabel('perpetual R per trade'); ax.set_ylabel(k); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Exports

In [ ]:
XP.to_csv(OUT / '06_trades.csv', index=False, float_format='%.6g'); PT.to_csv(OUT / '06_per_trade.csv', float_format='%.6g')
SENS.to_csv(OUT / '06_spread_sensitivity.csv', float_format='%.6g'); CUR.to_csv(OUT / '06_curves_summary.csv', float_format='%.6g')
pd.DataFrame(CURVE).to_csv(OUT / '06_curves_daily.csv', float_format='%.5g'); YR.to_csv(OUT / '06_years.csv', float_format='%.6g')
pd.DataFrame({**Q, **MD}).T.to_csv(OUT / '06_tests.csv', float_format='%.6g')
(OUT / '06_decision.json').write_text(json.dumps({'decision': DECISION, 'tests': {**Q, **MD}, 'registration': to.registration(O)}, indent=1, default=float))
(OUT / '06_manifest.json').write_text(json.dumps({'module': 'Turtle_Breakout_06', 'version': to.VERSION, 'synthetic': SYNTHETIC,
    'runtime_s': round(time.time() - T0, 1)}, indent=1))
print(sorted(p.name for p in OUT.iterdir()))